# EU Market Pulse — Pipeline Connection Test

## Objective

Verify that Python can connect securely to the Supabase (Postgres) database 
using credentials stored in a local `.env` file, before building the full 
automated update pipeline.

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv('../.env', override=True)
database_url = os.getenv('DATABASE_URL')

print(f"URL carregada: {database_url is not None}")

engine = create_engine(database_url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM countries;"))
    count = result.scalar()
    print(f"Connection successful. 'countries' table has {count} rows.")

URL carregada: True
Connection successful. 'countries' table has 19 rows.


## Connection Test Result

Successfully connected to the Supabase Postgres database using the Session 
Pooler connection string (required due to IPv6 connectivity issues with the 
Direct Connection on this network). The `countries` table returned 19 rows, 
confirming the database is accessible and ready for the automated update 
pipeline.

**Note:** the connection string is stored in a local `.env` file (excluded 
from version control via `.gitignore`) and loaded via `python-dotenv`.

## Test: Fetch Data Directly from Eurostat API

Before building the full automation script, we validate that we can fetch 
the same datasets used in the manual download (`tec00118`, `tec00120`, 
`tipsho10`) directly via the Eurostat API, using the `eurostat` Python package.

In [2]:
import eurostat

# Fetch the HICP inflation rate dataset directly from Eurostat's API
df_test = eurostat.get_data_df('tec00118')

print(df_test.shape)
df_test.head()

(42, 16)


,freq,unit,coicop18,geo\TIME_PERIOD,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,RCH_A_AVG,TOTAL,AL,NaN,NaN,NaN,3.2,1.8,1.7,2.2,2.3,6.6,5.3,2.6,2.3
1,A,RCH_A_AVG,TOTAL,AT,1.5,0.8,1.0,2.2,2.1,1.5,1.4,2.8,8.6,7.7,2.9,3.6
2,A,RCH_A_AVG,TOTAL,BE,0.5,0.6,1.8,2.2,2.3,1.3,0.4,3.2,10.3,2.3,4.3,3.0
3,A,RCH_A_AVG,TOTAL,BG,-1.6,-1.1,-1.3,1.2,2.6,2.5,1.2,2.8,13.0,8.6,2.6,3.5
4,A,RCH_A_AVG,TOTAL,CH,0.0,-0.8,-0.5,0.6,0.9,0.4,-0.8,0.5,2.7,2.3,1.1,0.1


In [3]:
print(df_test.columns.tolist())

['freq', 'unit', 'coicop18', 'geo\\TIME_PERIOD', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


## Reshape: Wide to Long Format

The Eurostat API returns data in wide format (one column per year). We 
reshape it to long format (one row per country-year) to match our database 
schema, using the same country scope and cleaning logic validated in the 
manual pipeline.

In [4]:
# Rename the geo column
df_test_clean = df_test.rename(columns={'geo\\TIME_PERIOD': 'country_code'})

# Identify year columns
year_columns = [col for col in df_test_clean.columns if col.isdigit()]

# Reshape from wide to long format
df_long = df_test_clean.melt(
    id_vars=['country_code'],
    value_vars=year_columns,
    var_name='year',
    value_name='inflation_rate'
)
df_long['year'] = df_long['year'].astype(int)

# Eurostat uses non-standard codes for Greece (EL) and the United Kingdom (UK) —
# map them to standard ISO-2 BEFORE filtering to our country scope
EUROSTAT_CODE_MAP = {'EL': 'GR', 'UK': 'GB'}
df_long['country_code'] = df_long['country_code'].replace(EUROSTAT_CODE_MAP)

# Our 19-country scope, by standard ISO-2 code
COUNTRY_CODES = ['GB', 'CH', 'IE', 'DE', 'FR', 'NL', 'AT', 'BE', 
                  'PT', 'ES', 'IT', 'GR', 'PL', 'RO', 'HU', 'CZ', 
                  'SE', 'DK', 'FI']

df_long = df_long[df_long['country_code'].isin(COUNTRY_CODES)].dropna(subset=['inflation_rate'])
df_long = df_long.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Shape: {df_long.shape}")
print(f"Countries found: {df_long['country_code'].nunique()} / {len(COUNTRY_CODES)}")

Shape: (222, 3)
Countries found: 19 / 19


In [5]:
missing = set(COUNTRY_CODES) - set(df_long['country_code'].unique())
print(f"Países em falta: {missing}")

Países em falta: set()


In [6]:
# Eurostat uses non-standard codes for Greece (EL) and the United Kingdom (UK)
# We map them to standard ISO-2 codes to stay consistent with our database schema
EUROSTAT_CODE_MAP = {'EL': 'GR', 'UK': 'GB'}

df_long['country_code'] = df_long['country_code'].replace(EUROSTAT_CODE_MAP)
df_long = df_long[df_long['country_code'].isin(COUNTRY_CODES)]

print(f"Countries found: {df_long['country_code'].nunique()} / {len(COUNTRY_CODES)}")

Countries found: 19 / 19


## Fetch: Price Level Index (tec00120)

Repeating the same fetch-and-clean process for the second dataset, following 
the identical pattern validated for inflation rate.

In [7]:
df_price_raw = eurostat.get_data_df('tec00120')

print(df_price_raw.shape)
print(df_price_raw.columns.tolist())
df_price_raw.head()

(43, 16)
['freq', 'indic_ppp', 'ppp_cat18', 'geo\\TIME_PERIOD', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


,freq,indic_ppp,ppp_cat18,geo\TIME_PERIOD,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,PLI_EU27_2020,E011,AL,50.1,48.3,51.1,52.4,55.9,58.3,59.4,61.6,61.9,65.2,70.6,72.6
1,A,PLI_EU27_2020,E011,AT,109.3,109.3,110.4,111.6,112.1,113.2,111.7,108.5,107.9,111.5,111.7,113.0
2,A,PLI_EU27_2020,E011,BA,54.5,53.8,53.3,54.5,54.8,54.8,56.1,56.2,58.4,58.4,58.4,59.2
3,A,PLI_EU27_2020,E011,BE,111.2,111.2,113.3,114.3,114.7,114.9,113.6,115.5,117.4,116.9,116.7,116.2
4,A,PLI_EU27_2020,E011,BG,49.3,49.7,49.7,51.2,51.9,53.6,55.2,55.5,58.2,59.2,60.5,62.5


In [8]:
# Rename the geo column
df_price_clean = df_price_raw.rename(columns={'geo\\TIME_PERIOD': 'country_code'})

# Identify year columns
year_columns = [col for col in df_price_clean.columns if col.isdigit()]

# Reshape from wide to long format
df_price_long = df_price_clean.melt(
    id_vars=['country_code'],
    value_vars=year_columns,
    var_name='year',
    value_name='price_level_index'
)
df_price_long['year'] = df_price_long['year'].astype(int)

# Apply the same Eurostat code mapping (Greece: EL->GR, UK: UK->GB)
df_price_long['country_code'] = df_price_long['country_code'].replace(EUROSTAT_CODE_MAP)

# Filter to our country scope
df_price_long = df_price_long[df_price_long['country_code'].isin(COUNTRY_CODES)].dropna(subset=['price_level_index'])
df_price_long = df_price_long.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Shape: {df_price_long.shape}")
print(f"Countries found: {df_price_long['country_code'].nunique()} / {len(COUNTRY_CODES)}")

Shape: (228, 3)
Countries found: 19 / 19


## Fetch: House Price Index (tipsho10)

Repeating the same process for the third dataset. Note: this dataset is 
known to exclude Switzerland and the United Kingdom (confirmed during manual 
data collection), so we expect fewer than 19 countries here — that is 
expected, not an error.

In [9]:
df_house_raw = eurostat.get_data_df('tipsho10')

print(df_house_raw.shape)
print(df_house_raw.columns.tolist())
df_house_raw.head()

(60, 29)
['freq', 'unit', 'geo\\TIME_PERIOD', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


,freq,unit,geo\TIME_PERIOD,2000,2001,2002,2003,2004,2005,2006,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,I15_A_AVG,AT,80.00,78.76,78.43,77.46,74.73,83.32,84.87,...,105.27,108.56,112.62,117.33,124.93,136.59,141.47,127.10,122.59,122.43
1,A,I15_A_AVG,BE,65.15,66.73,70.23,74.04,78.50,86.41,92.33,...,100.82,102.32,103.28,105.59,109.12,113.38,108.18,104.17,105.70,106.62
2,A,I15_A_AVG,BG,62.60,59.48,52.34,58.03,82.82,105.76,118.73,...,105.23,109.29,113.90,118.24,124.36,127.46,125.03,127.01,140.95,154.06
3,A,I15_A_AVG,CY,87.05,86.37,90.37,100.28,108.66,114.96,124.10,...,101.77,105.47,102.72,110.94,118.19,114.85,113.49,119.63,126.26,131.41
4,A,I15_A_AVG,CZ,65.64,69.44,77.27,85.00,82.57,83.72,88.01,...,106.00,114.95,121.33,128.57,135.42,155.63,159.03,144.13,146.79,157.35


In [10]:
# Rename the geo column
df_house_clean = df_house_raw.rename(columns={'geo\\TIME_PERIOD': 'country_code'})

# Identify year columns
year_columns = [col for col in df_house_clean.columns if col.isdigit()]

# Reshape from wide to long format
df_house_long = df_house_clean.melt(
    id_vars=['country_code'],
    value_vars=year_columns,
    var_name='year',
    value_name='house_price_index'
)
df_house_long['year'] = df_house_long['year'].astype(int)

# Apply the same Eurostat code mapping
df_house_long['country_code'] = df_house_long['country_code'].replace(EUROSTAT_CODE_MAP)

# Match the same scope used in the manual pipeline: 2016 onwards
df_house_long = df_house_long[df_house_long['year'] >= 2016]

# Filter to our country scope
df_house_long = df_house_long[df_house_long['country_code'].isin(COUNTRY_CODES)].dropna(subset=['house_price_index'])
df_house_long = df_house_long.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Shape: {df_house_long.shape}")
print(f"Countries found: {df_house_long['country_code'].nunique()} / {len(COUNTRY_CODES)}")

Shape: (340, 3)
Countries found: 17 / 19


In [11]:
print(df_house_raw['unit'].unique())


['I15_A_AVG' 'RCH_A_AVG']


In [12]:
# This dataset contains two metrics per country: the index (I15_A_AVG) and 
# the year-over-year % change (RCH_A_AVG). We only want the index, matching 
# the manually downloaded data used in the EDA notebook.
df_house_filtered = df_house_raw[df_house_raw['unit'] == 'I15_A_AVG'].copy()

# Rename the geo column
df_house_clean = df_house_filtered.rename(columns={'geo\\TIME_PERIOD': 'country_code'})

# Identify year columns
year_columns = [col for col in df_house_clean.columns if col.isdigit()]

# Reshape from wide to long format
df_house_long = df_house_clean.melt(
    id_vars=['country_code'],
    value_vars=year_columns,
    var_name='year',
    value_name='house_price_index'
)
df_house_long['year'] = df_house_long['year'].astype(int)

# Apply the same Eurostat code mapping
df_house_long['country_code'] = df_house_long['country_code'].replace(EUROSTAT_CODE_MAP)

# Match the same scope used in the manual pipeline: 2016 onwards
df_house_long = df_house_long[df_house_long['year'] >= 2016]

# Filter to our country scope
df_house_long = df_house_long[df_house_long['country_code'].isin(COUNTRY_CODES)].dropna(subset=['house_price_index'])
df_house_long = df_house_long.sort_values(['country_code', 'year']).reset_index(drop=True)

print(f"Shape: {df_house_long.shape}")
print(f"Countries found: {df_house_long['country_code'].nunique()} / {len(COUNTRY_CODES)}")

Shape: (170, 3)
Countries found: 17 / 19
